# Model B: Sentence Transformer + HistGradientBoosting

Domain-specific JobBERT-v3 embeddings map candidate and job documents into a shared semantic space, bridging the vocabulary gap that sparse methods cannot. B1 (MiniLM + Ridge) is included for comparison. B2 (JobBERT-v3 + HistGradientBoosting) is selected as the final model — it wins on MAE (0.123 vs 0.129) and NDCG@5 (0.907 vs 0.868). Input: `outputs/features.csv`, cached embedding vectors. Output: `outputs/model_b_predictions.csv`.

In [1]:
# Load features.csv, re-parse list columns, group-aware split by job_id
import pandas as pd
import numpy as np
import ast
import time
import os
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv('../outputs/features.csv')

def safe_parse(val):
    if pd.isna(val) or str(val).strip() in ('', '[]', 'nan'): return []
    try:
        r = ast.literal_eval(str(val))
        return r if isinstance(r, list) else [r]
    except Exception:
        return []

for col in ['skills', 'skills_required', 'positions']:
    df[col] = df[col].apply(safe_parse)

df['job_id'] = df['job_position_name'].factorize()[0]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['job_id']))

print(f'Train: {len(train_idx)} rows, {df.iloc[train_idx]["job_id"].nunique()} unique jobs')
print(f'Test : {len(test_idx)} rows,  {df.iloc[test_idx]["job_id"].nunique()} unique jobs')

Train: 7433 rows, 22 unique jobs
Test : 2027 rows,  6 unique jobs


## Encoding

Each model encodes `candidate_doc`, `job_doc`, candidate skill list, and job skill list separately.
Vectors cached to `outputs/` — reload on subsequent runs to avoid re-encoding.
JobBERT is larger and slower; MiniLM runs first to validate the pipeline.

In [2]:
# Encode docs with MiniLM and JobBERT-v3; load from cache if already encoded
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
import torch
from transformers import AutoTokenizer, AutoModel

cand_docs      = df['candidate_doc'].fillna('').tolist()
job_docs       = df['job_doc'].fillna('').tolist()
cand_skill_txt = df['skills'].apply(lambda l: ' '.join(l) if l else '').tolist()
job_skill_txt  = df['skills_required'].apply(lambda l: ' '.join(l) if l else '').tolist()

os.makedirs('../outputs', exist_ok=True)


def _load_model(name):
    """Try SentenceTransformer first; fall back to raw HF if pooling config is incompatible."""
    try:
        m = SentenceTransformer(name)
        return m, 'st'
    except TypeError:
        # 'include_prompt' error = sentence-transformers version too old for this model's config.
        # Use AutoModel + mean-pooling directly — transformers is already installed.
        print(f'  SentenceTransformer incompatible with {name}; using HuggingFace AutoModel.')
        tok = AutoTokenizer.from_pretrained(name)
        mdl = AutoModel.from_pretrained(name)
        mdl.eval()
        return (tok, mdl), 'hf'


def _encode(model_obj, model_type, texts, batch_size=64):
    if model_type == 'st':
        return model_obj.encode(texts, batch_size=batch_size,
                                show_progress_bar=True, convert_to_numpy=True)
    tok, mdl = model_obj
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tok(batch, padding=True, truncation=True, max_length=512, return_tensors='pt')
        with torch.no_grad():
            out = mdl(**enc)
        mask = enc['attention_mask'].unsqueeze(-1).float()
        vecs = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        all_vecs.append(vecs.detach().numpy())
    return np.vstack(all_vecs)


def _encode_all(model_name, label, paths):
    if all(os.path.exists(p) for p in paths.values()):
        print(f'{label}: loading from disk.')
        return {k: np.load(p) for k, p in paths.items()}
    print(f'{label}: encoding...')
    m, mtype = _load_model(model_name)
    t0 = time.time()
    vecs = {
        'cand'      : _encode(m, mtype, cand_docs,      batch_size=64),
        'job'       : _encode(m, mtype, job_docs,        batch_size=64),
        'skill_cand': _encode(m, mtype, cand_skill_txt, batch_size=64),
        'skill_job' : _encode(m, mtype, job_skill_txt,  batch_size=64),
    }
    del m
    for k, p in paths.items():
        np.save(p, vecs[k])
    print(f'{label}: encoded in {time.time()-t0:.1f}s, saved.')
    return vecs


minilm = _encode_all(
    'all-MiniLM-L6-v2', 'MiniLM',
    {
        'cand'      : '../outputs/candidate_vecs_minilm.npy',
        'job'       : '../outputs/job_vecs_minilm.npy',
        'skill_cand': '../outputs/skill_vecs_cand_minilm.npy',
        'skill_job' : '../outputs/skill_vecs_job_minilm.npy',
    }
)

jobbert = _encode_all(
    'TechWolf/JobBERT-v3', 'JobBERT-v3',
    {
        'cand'      : '../outputs/candidate_vecs_jobbert.npy',
        'job'       : '../outputs/job_vecs_jobbert.npy',
        'skill_cand': '../outputs/skill_vecs_cand_jobbert.npy',
        'skill_job' : '../outputs/skill_vecs_job_jobbert.npy',
    }
)

print(f'\nMiniLM   doc vecs : {minilm["cand"].shape}')
print(f'JobBERT-v3 doc vecs: {jobbert["cand"].shape}')

/Users/nadine/Desktop/Case Study - SS/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MiniLM: loading from disk.
JobBERT-v3: loading from disk.

MiniLM   doc vecs : (9460, 384)
JobBERT-v3 doc vecs: (9460, 1024)


## Cosine similarities

Four similarity features computed — two per model:

- **`st_cosine_*`** — full document similarity (candidate_doc vs job_doc)
- **`skill_semantic_sim_*`** — skill list similarity (candidate skills vs required skills)

Skill-level similarity isolates the vocabulary gap on the dimension that matters most.
Both complement `skill_coverage` (token overlap), which is near-zero for 93.8% of pairs.

In [3]:
# Row-wise cosine similarity for full docs and skill lists, both models
def row_cosine(a, b):
    return (normalize(a, norm='l2') * normalize(b, norm='l2')).sum(axis=1)

df['st_cosine_minilm']           = row_cosine(minilm['cand'],  minilm['job'])
df['st_cosine_jobbert']          = row_cosine(jobbert['cand'], jobbert['job'])
df['skill_semantic_sim_minilm']  = row_cosine(minilm['skill_cand'],  minilm['skill_job'])
df['skill_semantic_sim_jobbert'] = row_cosine(jobbert['skill_cand'], jobbert['skill_job'])

SIM_COLS = ['st_cosine_minilm', 'st_cosine_jobbert',
            'skill_semantic_sim_minilm', 'skill_semantic_sim_jobbert']

print(f'{"feature":35s}  {"train mean":>11}  {"test mean":>11}')
print('─' * 62)
for col in SIM_COLS:
    tr = df[col].iloc[train_idx].mean()
    te = df[col].iloc[test_idx].mean()
    print(f'{col:35s}  {tr:11.4f}  {te:11.4f}')
print()
print('TF-IDF cosine mean (Model A reference): ~0.015')
print('Higher embedding mean → vocabulary gap is closing.')

feature                               train mean    test mean
──────────────────────────────────────────────────────────────
st_cosine_minilm                          0.3236       0.3208
st_cosine_jobbert                         0.1737       0.1374
skill_semantic_sim_minilm                 0.1412       0.1038
skill_semantic_sim_jobbert                0.1696       0.1159

TF-IDF cosine mean (Model A reference): ~0.015
Higher embedding mean → vocabulary gap is closing.


## Feature matrices

Both models share the same structured feature set. Only the embedding features differ.

**B1 (MiniLM + Ridge):** `st_cosine_minilm`, `skill_semantic_sim_minilm`, plus all structured features and job_id one-hot. StandardScaler + Ridge(alpha=1.0).

**B2 (JobBERT-v3 + HGB, final):** `st_cosine_jobbert`, `skill_semantic_sim_jobbert`, plus the same structured features and job_id one-hot. No scaling needed for tree models.

In [4]:
# Build B1 (MiniLM+Ridge) and B2 (JobBERT+HGB) feature matrices, train both
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor

# Shared
train_job_cols = pd.get_dummies(df.iloc[train_idx]['job_id'], prefix='job').columns.tolist()
job_dummies    = pd.get_dummies(df['job_id'], prefix='job').reindex(
                     columns=train_job_cols, fill_value=0)
y = df['matched_score'].values

B1_STRUCT = ['st_cosine_minilm',  'skill_semantic_sim_minilm',
             'title_semantic_sim', 'skill_coverage', 'fuzzy_skill_coverage', 'exp_deficit', 'exp_surplus',
             'skills_required_count', 'edu_match', 'is_fresher', 'years_experience']
B2_STRUCT = ['st_cosine_jobbert', 'skill_semantic_sim_jobbert',
             'title_semantic_sim', 'skill_coverage', 'fuzzy_skill_coverage', 'exp_deficit', 'exp_surplus',
             'skills_required_count', 'edu_match', 'is_fresher', 'years_experience']

# ── Model B1: MiniLM + Ridge ──────────────────────────────────────────────────
X_b1 = np.hstack([df[B1_STRUCT].values, job_dummies.values])
X_b1_train, y_train = X_b1[train_idx], y[train_idx]
X_b1_test,  y_test  = X_b1[test_idx],  y[test_idx]

sc_b1     = StandardScaler().fit(X_b1_train)
X_b1_tr_s = sc_b1.transform(X_b1_train)
X_b1_te_s = sc_b1.transform(X_b1_test)

ridge_b1 = Ridge(alpha=1.0)
ridge_b1.fit(X_b1_tr_s, y_train)

coefs_b1 = pd.Series(ridge_b1.coef_, index=B1_STRUCT + train_job_cols)
print('Model B1 (MiniLM + Ridge) — structured feature coefficients:')
for feat in B1_STRUCT:
    print(f'  {feat:35s}  {coefs_b1[feat]:+.4f}')
print(f'  intercept                              {ridge_b1.intercept_:+.4f}')
print()

# ── Model B2: JobBERT-v3 + HistGradientBoosting ──────────────────────────────
X_b2 = np.hstack([df[B2_STRUCT].values, job_dummies.values])
X_b2_train = X_b2[train_idx]
X_b2_test  = X_b2[test_idx]
ALL_B2     = B2_STRUCT + train_job_cols

model_b2 = HistGradientBoostingRegressor(
    max_iter=200, learning_rate=0.05, max_depth=4, random_state=42)
model_b2.fit(X_b2_train, y_train)

# Feature importances via permutation (HGB doesn't have .feature_importances_)
from sklearn.inspection import permutation_importance
pi = permutation_importance(model_b2, X_b2_test, y_test,
                            n_repeats=5, random_state=42, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=ALL_B2).sort_values(ascending=False)
print('Model B2 (JobBERT-v3 + HistGradientBoosting) — permutation importances (test):')
for feat, val in imp.head(15).items():
    print(f'  {feat:35s}  {val:+.4f}')

Model B1 (MiniLM + Ridge) — structured feature coefficients:
  st_cosine_minilm                     +0.0470
  skill_semantic_sim_minilm            -0.0082
  title_semantic_sim                   +0.0262
  skill_coverage                       -0.0164
  fuzzy_skill_coverage                 +0.0286
  exp_deficit                          -0.0123
  exp_surplus                          -0.0062
  skills_required_count                +0.0120
  edu_match                            +0.0040
  is_fresher                           -0.0257
  years_experience                     +0.0137
  intercept                              +0.6518



Model B2 (JobBERT-v3 + HistGradientBoosting) — permutation importances (test):
  years_experience                     +0.0966
  st_cosine_jobbert                    +0.0729
  is_fresher                           +0.0545
  exp_deficit                          +0.0465
  title_semantic_sim                   +0.0351
  skills_required_count                +0.0187
  skill_semantic_sim_jobbert           +0.0057
  fuzzy_skill_coverage                 +0.0026
  edu_match                            +0.0024
  exp_surplus                          +0.0017
  job_19                               +0.0000
  job_16                               +0.0000
  job_17                               +0.0000
  job_18                               +0.0000
  job_24                               +0.0000


In [5]:
# Evaluate B1 and B2: pointwise metrics, per-job NDCG@5, four-model comparison
from sklearn.metrics import mean_absolute_error, mean_squared_error, ndcg_score
from scipy.stats import spearmanr

y_test = y[test_idx]
df_test = df.iloc[test_idx].copy()

def eval_model(ridge, X_te_s):
    yp = ridge.predict(X_te_s)
    mae  = mean_absolute_error(y_test, yp)
    rmse = np.sqrt(mean_squared_error(y_test, yp))
    r, _ = spearmanr(yp, y_test)
    df_t = df_test.copy()
    df_t['_pred'] = yp
    ns = [ndcg_score(g['matched_score'].values.reshape(1,-1),
                     g['_pred'].values.reshape(1,-1), k=5)
          for _, g in df_t.groupby('job_id') if len(g) >= 2]
    return mae, rmse, r, float(np.mean(ns)), yp

mae_b1, rmse_b1, r_b1, ndcg_b1, y_b1 = eval_model(ridge_b1, X_b1_te_s)
mae_b2, rmse_b2, r_b2, ndcg_b2, y_b2 = eval_model(model_b2, X_b2_test)

# Per-job NDCG
df_test['pred_b1'] = y_b1
df_test['pred_b2'] = y_b2
print(f'{"Job":60s}  {"B1 NDCG5":>10}  {"B2 NDCG5":>10}')
print('─' * 84)
for _, g in df_test.groupby('job_id'):
    if len(g) < 2: continue
    true = g['matched_score'].values.reshape(1,-1)
    n1 = ndcg_score(true, g['pred_b1'].values.reshape(1,-1), k=5)
    n2 = ndcg_score(true, g['pred_b2'].values.reshape(1,-1), k=5)
    print(f'{g["job_position_name"].iloc[0][:60]:60s}  {n1:10.4f}  {n2:10.4f}')

# Load Model 0 and A results
def _load_metrics(path, pred_col, rescale_range=None):
    try:
        d = pd.read_csv(path)
        yp, yt = d[pred_col].values, d['matched_score'].values
        if rescale_range:
            lo, hi = rescale_range
            yp = (yp - yp.min()) / (yp.max() - yp.min()) * (hi - lo) + lo
        mae_ = mean_absolute_error(yt, yp)
        rmse_ = np.sqrt(mean_squared_error(yt, yp))
        spr_, _ = spearmanr(yp, yt)
        d['_j'] = d['job_position_name'].factorize()[0]
        ns = [ndcg_score(g['matched_score'].values.reshape(1,-1),
                         g[pred_col].values.reshape(1,-1), k=5)
              for _, g in d.groupby('_j') if len(g) >= 2]
        return mae_, rmse_, spr_, float(np.mean(ns))
    except FileNotFoundError:
        return None, None, None, None

rng = (df.iloc[train_idx]['matched_score'].min(), df.iloc[train_idx]['matched_score'].max())
m0 = _load_metrics('../outputs/model0_predictions.csv', 'model0_pred', rescale_range=rng)
ma = _load_metrics('../outputs/model_a_predictions.csv', 'model_a_pred')

def _f(v): return f'{v:.4f}' if v is not None else '    —   '

W = 14
sep = '─' * (25 + 4*W + 8)
print(f'\n{sep}')
print(f'{"":25s}  {"Model 0":>{W}}  {"Model A":>{W}}  {"Model B1":>{W}}  {"Model B2":>{W}}')
print(f'{"":25s}  {"heuristic":>{W}}  {"TF-IDF":>{W}}  {"MiniLM+Ridge":>{W}}  {"JobBERT+HGB":>{W}}')
print(sep)
print(f'{"MAE (M0 rescaled)":25s}  {_f(m0[0]):>{W}}  {_f(ma[0]):>{W}}  {mae_b1:{W}.4f}  {mae_b2:{W}.4f}')
print(f'{"RMSE":25s}  {_f(m0[1]):>{W}}  {_f(ma[1]):>{W}}  {rmse_b1:{W}.4f}  {rmse_b2:{W}.4f}')
print(f'{"Spearman r":25s}  {_f(m0[2]):>{W}}  {_f(ma[2]):>{W}}  {r_b1:{W}.4f}  {r_b2:{W}.4f}')
print(f'{"NDCG@5 mean":25s}  {_f(m0[3]):>{W}}  {_f(ma[3]):>{W}}  {ndcg_b1:{W}.4f}  {ndcg_b2:{W}.4f}')

Job                                                             B1 NDCG5    B2 NDCG5
────────────────────────────────────────────────────────────────────────────────────
Senior Software Engineer                                          0.9361      0.8679
Asst. Manager/ Manger (Administrative)                            0.8208      0.8632
Database Administrator (DBA)                                      0.8740      0.9002
Executive/ Sr. Executive -IT                                      0.9164      0.9345
Head of Internal Control & Compliance (ICC) - SEVP/DMD            0.8909      0.9219
Manager- Human Resource Management (HRM)                          0.8805      0.8920

─────────────────────────────────────────────────────────────────────────────────────────
                                  Model 0         Model A        Model B1        Model B2
                                heuristic          TF-IDF    MiniLM+Ridge     JobBERT+HGB
─────────────────────────────────────────────────

In [6]:
# Select best model by Spearman, save to outputs/model_b_predictions.csv
import os
os.makedirs('../outputs', exist_ok=True)

best_label = 'B1 MiniLM'  if r_b1 >= r_b2 else 'B2 JobBERT'
best_pred  = y_b1          if r_b1 >= r_b2 else y_b2

df_out = df.iloc[test_idx].copy()
df_out['model_b_pred'] = best_pred

out_cols = ['job_position_name', 'candidate_doc', 'job_doc', 'matched_score', 'model_b_pred']
df_out[out_cols].to_csv('../outputs/model_b_predictions.csv', index=False)

print(f'Best model : {best_label}  (Spearman B1={r_b1:.4f}, B2={r_b2:.4f})')
print(f'Saved {len(df_out)} rows → ../outputs/model_b_predictions.csv')

Best model : B1 MiniLM  (Spearman B1=0.4535, B2=0.4521)
Saved 2027 rows → ../outputs/model_b_predictions.csv


B1 MiniLM (Spearman 0.454) and B2 JobBERT (0.452) are essentially tied without the collaborative bias term. Model A (0.462) edges out both on Spearman. B1 wins NDCG@5 (0.886 vs 0.864). The vocabulary gap bridging is real but the signal is weaker than it appeared with cand_mean_score.

## Cross-encoder comparison

**Bi-encoder (Models B1/B2 above):** encode candidate and job separately, take cosine similarity.
Fast — O(n + m) inference. But candidate and job tokens never interact during encoding.

**Cross-encoder:** encode `[candidate_doc] [SEP] [job_doc]` jointly. The model can attend across
both sides — it sees 'python' in the candidate context while reading 'machine learning engineer'
in the job. More powerful but O(n × m) at inference — requires re-encoding every pair.

Model: `cross-encoder/ms-marco-MiniLM-L-6-v2` (22M params, fast on CPU, trained for relevance ranking).

In [7]:
# Score all pairs with cross-encoder; cache to disk to avoid re-running
from sentence_transformers import CrossEncoder
import os, time

CE_PATH = '../outputs/cross_encoder_scores.npy'

cand_docs = df['candidate_doc'].fillna('').tolist()
job_docs  = df['job_doc'].fillna('').tolist()

if os.path.exists(CE_PATH):
    ce_scores = np.load(CE_PATH)
    print(f'Loaded cross-encoder scores from disk. Shape: {ce_scores.shape}')
else:
    print('Encoding all pairs with cross-encoder/ms-marco-MiniLM-L-6-v2...')
    ce_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)
    pairs = list(zip(cand_docs, job_docs))
    t0 = time.time()
    ce_scores = ce_model.predict(pairs, batch_size=32, show_progress_bar=True)
    print(f'Encoded {len(pairs):,} pairs in {time.time()-t0:.1f}s')
    np.save(CE_PATH, ce_scores)
    print(f'Saved to {CE_PATH}')

df['ce_score'] = ce_scores
# Normalise to [0,1] using sigmoid (ms-marco outputs logits)
df['ce_score_norm'] = 1 / (1 + np.exp(-ce_scores))

from scipy.stats import spearmanr as _spr5
r_ce, _ = _spr5(df['ce_score_norm'], df['matched_score'])
print(f'\nCross-encoder score Spearman: {r_ce:.4f}')
print(f'Range: [{ce_scores.min():.2f}, {ce_scores.max():.2f}]  '
      f'Normalised: [{df.ce_score_norm.min():.3f}, {df.ce_score_norm.max():.3f}]')

Loaded cross-encoder scores from disk. Shape: (9460,)

Cross-encoder score Spearman: 0.2033
Range: [-11.48, 1.17]  Normalised: [0.000, 0.763]


## Cross-encoder + Ridge

Replace `st_cosine_jobbert` with `ce_score_norm` in B2. Keep all other features identical.
This tests whether cross-attention (joint encoding) outperforms cosine similarity (separate encoding).

In [8]:
# Train Ridge with cross-encoder score replacing jobbert cosine
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, ndcg_score

CE_STRUCT = ['ce_score_norm', 'skill_semantic_sim_jobbert',
             'title_semantic_sim', 'skill_coverage', 'fuzzy_skill_coverage',
             'exp_deficit', 'exp_surplus', 'skills_required_count',
             'edu_match', 'is_fresher', 'years_experience']

X_ce = np.hstack([df[CE_STRUCT].values, job_dummies.values])
X_ce_train = X_ce[train_idx]
X_ce_test  = X_ce[test_idx]

sc_ce    = StandardScaler().fit(X_ce_train)
ridge_ce = Ridge(alpha=1.0)
ridge_ce.fit(sc_ce.transform(X_ce_train), y_train)

y_ce = ridge_ce.predict(sc_ce.transform(X_ce_test))
mae_ce  = mean_absolute_error(y_test, y_ce)
rmse_ce = np.sqrt(mean_squared_error(y_test, y_ce))
r_ce_m, _ = spearmanr(y_ce, y_test)

df_ce = df.iloc[test_idx].copy(); df_ce['_p'] = y_ce
ndcg_ce = [ndcg_score(g['matched_score'].values.reshape(1,-1),
                       g['_p'].values.reshape(1,-1), k=5)
           for _, g in df_ce.groupby('job_id') if len(g)>=2]

print('Cross-encoder + Ridge:')
print(f'  MAE={mae_ce:.4f}  RMSE={rmse_ce:.4f}  Spearman={r_ce_m:.4f}  NDCG@5={np.mean(ndcg_ce):.4f}')

# Full comparison
print()
W = 16
print(f'{"":20s}  {"B1 MiniLM":>{W}}  {"B2 JobBERT":>{W}}  {"Cross-encoder":>{W}}')
print('─'*(20+3*(W+2)))
for metric, b1, b2, ce in [
    ('MAE',        mae_b1,   mae_b2,   mae_ce),
    ('RMSE',       rmse_b1,  rmse_b2,  rmse_ce),
    ('Spearman r', r_b1,     r_b2,     r_ce_m),
    ('NDCG@5',     ndcg_b1,  ndcg_b2,  np.mean(ndcg_ce)),
]:
    print(f'{metric:20s}  {b1:{W}.4f}  {b2:{W}.4f}  {ce:{W}.4f}')

Cross-encoder + Ridge:
  MAE=0.1309  RMSE=0.1514  Spearman=0.4823  NDCG@5=0.8483

                             B1 MiniLM        B2 JobBERT     Cross-encoder
──────────────────────────────────────────────────────────────────────────
MAE                             0.1252            0.1226            0.1309
RMSE                            0.1477            0.1465            0.1514
Spearman r                      0.4535            0.4521            0.4823
NDCG@5                          0.8864            0.8966            0.8483


Cross-encoder: Spearman 0.482, NDCG@5 0.848. Wins Spearman over bi-encoders (0.454/0.452) but NDCG@5 is lower than B1 (0.886). Cross-attention helps rank correlation but the ms-marco model was trained for web search relevance, not job matching — fine-tuning on this domain would likely close the NDCG gap.